# EJEMPLO RL CON ATARI

````
$ pip install "gymnasium[atari]"
$ pip install autorom[accept-rom-license]
$ AutoROM --accept-license
$ apt-get install cmake
$ apt-get install python3-dev
$ pip install atari-py
$ pip install moviepy==1.0.3
````

In [ ]:
import gymnasium as gym
import ale_py
env = gym.make("ALE/Breakout-v5")

In [ ]:
epochs = 0

frames = []  # for animation
done = False

env = gym.make("ALE/Breakout-v5", render_mode="rgb_array")
observation, info = env.reset()

while not done:
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)

    # Put each rendered frame into dict for animation
    frames.append(
        {
            "frame": env.render(),
            "state": observation,
            "action": action,
            "reward": reward,
        }
    )

    epochs += 1
    if epochs == 1000:
        break

In [ ]:
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.animation as animation

def animate_frames(frames):
    fig, ax = plt.subplots()
    ax.axis('off')
    
    img = ax.imshow(frames[0]["frame"])
    
    def update(i):
        img.set_data(frames[i]["frame"])
        ax.set_title(f"Step: {i} | Action: {frames[i]['action']} | Reward: {frames[i]['reward']}")
        return [img]
    
    anim = animation.FuncAnimation(
        fig, 
        update, 
        frames=len(frames), 
        interval=50,  # ms entre frames (50ms = 20fps)
        blit=True
    )
    
    plt.close(fig)  # Evita mostrar figura estática
    return HTML(anim.to_jshtml())  # Renderiza como widget interactivo

animate_frames(frames)

In [ ]:
# !pip install moviepy

from moviepy import ImageSequenceClip
 

def create_gif(frames: dict, filename, fps=100):
    """
    Creates a GIF animation from a list of RGBA NumPy arrays.

    Args:
        frames: A list of RGBA NumPy arrays representing the animation frames.
        filename: The output filename for the GIF animation.
        fps: The frames per second of the animation (default: 10).
    """
    rgba_frames = [frame["frame"] for frame in frames]

    clip = ImageSequenceClip(rgba_frames, fps=fps)
    clip.write_gif(filename, fps=fps)

# Example usage
create_gif(frames, "animation.gif") #saves the GIF locally


In [ ]:
# Import the gymnasium library
import gymnasium as gym

# Create the environment
env = gym.make('MountainCar', render_mode='rgb_array')

# Get the initial state
initial_state, info = env.reset(seed=42)

position = initial_state[0]
velocity = initial_state[1]

print(f"The position of the car along the x-axis is {position} (m)")
print(f"The velocity of the car is {velocity} (m/s)")

In [ ]:
# Import the gymnasium library
import gymnasium as gym

import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from random import randint

env = gym.make('MountainCar-v0', render_mode='rgb_array')
initial_state, _ = env.reset()

In [ ]:
import numpy as np
def discretizar(valor):
    valor_discreto = ((valor-env.observation_space.low)/(env.observation_space.high-env.observation_space.low))*20
    return tuple(valor_discreto.astype(np.int32)) # Pasamos a int32 por requisitos de gymnasium

In [ ]:
q_table = np.random.uniform(low=-1, high=1, size=[20,20,3]) # Creamos la tabla de estados (Q_table)

In [ ]:


def render(episode, total_reward):
    state_image = env.render()
    fig, ax = plt.subplots()
    ax.imshow(state_image)
    ax.axis('off')
    ax.set_title(f"Episodio {episode+1} | Recompensa acumulada: {total_reward:.1f}")
    display(fig)
    plt.close(fig)
    clear_output(wait=True)

learning_rate = 0.1
discount_factor = 0.05
episodes = 5000
reward_list =[]



## Ejecución del juego

In [ ]:
for episode in range(episodes):
    state = discretizar(env.reset()[0])
    final = False
    total_reward = 0
    while not final:
        if randint(0, 10)>2:
            action = np.argmax(q_table[state]) # 'mejor' movimiento (con mayor recompensa)
        else:
            action =randint(0,2) # movimiento aleatorio
        new_state, reward, final, truncated, info = env.step(action)
        final = final or truncated 
        
        q_table[state][action] = q_table[state][action] + learning_rate * (reward + discount_factor * np.max(q_table[discretizar(new_state)])) # fórmula Q Learning
        state = discretizar(new_state)
        total_reward += reward
        if (episode + 1)%500 == 0:
            render(episode, total_reward)
    reward_list.append(total_reward)
    if (episode + 1) % 100 == 0:
        print(f"Episodio {episode+1} - Recompensa: {np.mean(reward_list)}")
env.close()

# SPACE INVADERS (ATARI)

## 1.- Preparación del entorno

In [1]:
# Import the gymnasium library
import gymnasium as gym

In [2]:
import tensorflow as tf
import os

2026-02-26 17:58:41.942869: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-26 17:58:42.058509: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-26 17:58:45.925759: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
# Configuración GPU

os.environ['TF_GPU_ALLOCATOR'] = 'cuda_malloc_async'  # mejor rendimiento en RTX serie 40

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅ Entrenando en GPU: {gpus[0]}")
else:
    print("⚠️ GPU no detectada, entrenando en CPU (muy lento)")

✅ Entrenando en GPU: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')


In [4]:
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from random import randint
import numpy as np

In [ ]:
# !pip install ale-py

In [5]:
from ale_py import ALEInterface
ale = ALEInterface

In [6]:
env = gym.make('ALE/SpaceInvaders-v5', render_mode='rgb_array')
initial_state, _ = env.reset()
height, width, channels = env.observation_space.shape
actions = env.action_space 

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


In [7]:
env.unwrapped.get_action_meanings()

def render(episode, total_reward):
    state_image = env.render()
    fig, ax = plt.subplots()
    ax.imshow(state_image)
    ax.axis('off')
    ax.set_title(f"Episodio {episode+1} | Puntuación: {total_reward:.1f}")
    display(fig)
    plt.close(fig)
    clear_output(wait=True)

learning_rate = 0.1
discount_factor = 0.05
episodes = 5000
reward_list =[]



## 2.- Acciones

In [ ]:
import random


episodes = 5

for episode in range(episodes):
    total_reward = 0
    state = env.reset()
    final = False
    
    while not final:
        render(episode, total_reward)
        action = random.choice([0,1,2,3,4,5])
        new_state, reward, final, truncated, info = env.step(action)
        total_reward += reward
        reward_list.append(total_reward)
    print(f"Episodio {episode} - Puntuación: {np.mean(reward_list)}")
env.close()

        

## 3. - Crear modelo de Deep Q-Learning con Keras

In [8]:
from keras.models import Sequential
from keras.layers import Dense, Flatten, Convolution2D, Input
from keras.optimizers import Adam

def build_model(height, width, channels, actions):
    with tf.device('/GPU:0'): 
        model = Sequential()
        model.add(Input(shape=(height, width, channels)))
        model.add(Convolution2D(32, (8,8), strides=(4,4), activation='relu'))
        model.add(Convolution2D(64, (4,4), strides=(2,2), activation='relu'))
        model.add(Convolution2D(64, (3,3), activation='relu'))
        model.add(Flatten())
        model.add(Dense(512, activation='relu'))
        model.add(Dense(256, activation='relu'))
        model.add(Dense(actions, activation='linear'))
        model.compile(optimizer=Adam(learning_rate=1e-4), loss='mse')
    return model

In [9]:
model = build_model(height,width,channels, actions.n)
model.summary()

I0000 00:00:1772125160.837588  179990 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1772125160.839530  179990 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6096 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 51, 39, 32)     │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 18, 64)     │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 22, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 22528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │    11,534,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,743,654 (44.80 MB)

 Trainable params: 11,743,654 (44.80 MB)

 Non-trainable params: 0 (0.00 B)

## 4.- Construcción Agente con Keras.

In [10]:
from collections import deque
import random
import numpy as np

class DQNAgent:
    def __init__(self, model, actions, memory_limit=10000, window_length=3):
        self.model = model
        self.actions = actions
        
        # Memoria de repetición (Experience Replay):
        # guarda las últimas experiencias para entrenar en mini-batches
        # evita que el modelo aprenda correlaciones temporales
        self.memory = deque(maxlen=memory_limit)
        
        # Epsilon: controla exploración vs explotación
        # al inicio explora mucho (eps=1.0), gradualmente explota lo aprendido
        self.eps = 1.0
        self.eps_min = 0.1        # nunca deja de explorar un mínimo
        self.eps_decay = 0.9995   # velocidad de reducción de epsilon
        
        self.gamma = 0.99         # factor de descuento: cuánto importan recompensas futuras
        self.batch_size = 32      # tamaño del mini-batch para entrenamiento

    def remember(self, state, action, reward, next_state, done):
        """Guarda una experiencia en memoria: (s, a, r, s', done)"""
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        """
        Política epsilon-greedy:
        - Con probabilidad eps -> acción aleatoria (exploración)
        - Con probabilidad 1-eps -> mejor acción según el modelo (explotación)
        """
        if np.random.rand() <= self.eps:
            return random.randrange(self.actions)
        q_values = self.model.predict(state[np.newaxis], verbose=0)
        return np.argmax(q_values[0])

    def replay(self):
        if len(self.memory) < self.batch_size:
            return
        
        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        # Convertir a arrays NumPy para procesamiento vectorizado en GPU
        states      = np.array(states,      dtype=np.float32)
        next_states = np.array(next_states, dtype=np.float32)
        rewards     = np.array(rewards,     dtype=np.float32)
        dones       = np.array(dones,       dtype=np.float32)
        actions     = np.array(actions,     dtype=np.int32)

        # Predecir todo el batch de golpe en GPU (1 llamada en vez de 32)
        q_current   = self.model.predict(states,      verbose=0)
        q_next      = self.model.predict(next_states, verbose=0)

        # Ecuación de Bellman vectorizada
        targets = rewards + self.gamma * np.max(q_next, axis=1) * (1 - dones)
        
        # Actualizar solo los Q-valores de las acciones tomadas
        for i, action in enumerate(actions):
            q_current[i][action] = targets[i]

        # Un solo fit para todo el batch 👈 aquí se aprovecha la GPU
        self.model.fit(states, q_current, batch_size=self.batch_size, verbose=0)

        if self.eps > self.eps_min:
            self.eps *= self.eps_decay

    def save(self, filepath):
        """Guarda los pesos del modelo entrenado"""
        self.model.save_weights(filepath)
        print(f"Modelo guardado en {filepath}")

    def load(self, filepath):
        """Carga pesos de un modelo previamente entrenado"""
        self.model.load_weights(filepath)
        print(f"Modelo cargado desde {filepath}")
        
        

dqn = DQNAgent(model=model, actions=actions.n)


## 3. Entrenamiento del agente

In [20]:
import sys

def train_dqn(dqn, env, nb_steps=10000, visualize=False, verbose=2, save_path="dqn_spaceinvaders.weights.h5"): # original nb_steps=100000
    """
    Bucle principal de entrenamiento:
    - Interactúa con el entorno step a step
    - Guarda experiencias en memoria
    - Entrena el modelo con replay
    - Guarda el modelo cada 10000 pasos
    """
    step = 0
    episode = 0
    reward_list = []
    fig, ax = plt.subplots() if visualize else (None, None)

    while step < nb_steps:
        state, _ = env.reset()
        done = False
        total_reward = 0

        while not done and step < nb_steps:
            if visualize:
                ax.clear()
                ax.imshow(env.render())
                ax.axis('off')
                ax.set_title(f"Step {step}/{nb_steps} | Ep {episode} | Reward: {total_reward:.1f} | Eps: {dqn.eps:.3f}")
                display(fig)
                clear_output(wait=True)

            # 1. Elegir acción
            action = dqn.act(state)
            
            # 2. Ejecutar acción en el entorno
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            # 3. Guardar experiencia
            dqn.remember(state, action, reward, next_state, done)
            
            # 4. Entrenar con replay
            dqn.replay()

            state = next_state
            total_reward += reward
            step += 1

        episode += 1
        reward_list.append(total_reward)

        if verbose >= 2:
            progreso = (step / nb_steps) * 100
            barra = int(progreso / 2)
           # clear_output(wait=True)  # 👈 limpia el output anterior en Jupyter
            sys.stdout.write(
                f"\rEp {episode:4d} | "
                f"[{'█' * barra}{'░' * (50 - barra)}] {progreso:5.1f}% | "
                f"Steps: {step:,}/{nb_steps:,} | "
                f"Recompensa: {total_reward:6.1f} | "
                f"ε: {dqn.eps:.3f}"
            )
            sys.stdout.flush()
        
        # Guardar modelo cada 1000 pasos
        if step % 1000 == 0:  # original eran 10000 pasos
            dqn.save(save_path)

    if visualize:
        plt.close(fig)
    
    dqn.save(save_path)  # guardado final
    print("✅ Entrenamiento completado")
    return reward_list


In [12]:
# Ejecutar una vez antes de train_dqn
with tf.device('/GPU:0'):
    test = tf.constant([[1.0, 2.0]])
    print("✅ Operación de prueba en GPU correcta")

# Durante el entrenamiento verás en nvidia-smi que la GPU Memory y GPU Util suben

✅ Operación de prueba en GPU correcta


In [21]:
reward_list = train_dqn(dqn, env, nb_steps=10000, visualize=False, verbose=2) # nb_steps=100000

KeyboardInterrupt: 

## 4. Curva de aprendizaje

In [ ]:
def plot_training(reward_list, window=50):
    """
    Muestra la evolución de las recompensas durante el entrenamiento.
    La media móvil suaviza el ruido para ver la tendencia real.
    """
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(reward_list, alpha=0.4, color='blue', label='Recompensa por episodio')
    
    if len(reward_list) >= window:
        media_movil = np.convolve(reward_list, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(reward_list)), media_movil, 
                color='red', linewidth=2, label=f'Media móvil ({window} eps)')
    
    ax.set_xlabel('Episodio')
    ax.set_ylabel('Recompensa total')
    ax.set_title('Curva de aprendizaje - DQN SpaceInvaders')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_training(reward_list)

## 5. Test del agente entrenado

In [ ]:
def test_dqn(dqn, env, nb_episodes=5, visualize=True, weights_path="dqn_spaceinvaders.weights.h5"):
    """
    Testea el agente entrenado:
    - Carga los pesos guardados
    - Desactiva exploración (eps=0) para usar solo lo aprendido
    - Muestra el juego en pantalla
    """
    # Cargar modelo entrenado
    dqn.load(weights_path)
    
    # Desactivar exploración: solo explotación del conocimiento aprendido
    dqn.eps = 0.0

    fig, ax = plt.subplots()
    test_rewards = []

    for episode in range(nb_episodes):
        state, _ = env.reset()
        done = False
        total_reward = 0

        while not done:
            if visualize:
                ax.clear()
                ax.imshow(env.render())
                ax.axis('off')
                ax.set_title(f"TEST | Episodio {episode+1}/{nb_episodes} | Puntuación: {total_reward:.1f}")
                display(fig)
                clear_output(wait=True)

            # Solo explotación: siempre la mejor acción conocida
            action = dqn.act(state)
            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward

        test_rewards.append(total_reward)
        print(f"Episodio test {episode+1} | Puntuación: {total_reward:.1f}")

    plt.close(fig)
    print(f"\n🏆 Puntuación media en test: {np.mean(test_rewards):.1f}")
    print(f"   Mejor partida: {max(test_rewards):.1f}")
    print(f"   Peor partida:  {min(test_rewards):.1f}")
    
    return test_rewards


test_rewards = test_dqn(dqn, env, nb_episodes=5, visualize=True)